# Resumen Final del Proyecto Thoracolumbar Explicado - Colab

Este notebook consolida la historia tecnica del proyecto:

- objetivo
- datos y decisiones
- arquitectura seleccionada
- experimentos clave
- resultados comparativos
- argumentos de por que se eligio la solucion final

## Objetivo del proyecto

Construir un pipeline reproducible para identificar vertebras toracicas y lumbares
en radiografias, excluyendo la region cervical.

## 0. Preparacion de Colab

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception as exc:
    print('No se detecto entorno Colab o Drive ya estaba montado:', exc)

def resolve_project_root() -> Path:
    env_override = os.environ.get('PROJECT_ROOT_OVERRIDE', '').strip()
    candidates = []
    if env_override:
        candidates.append(Path(env_override))
    candidates.extend([
        Path('/content/drive/MyDrive/dataRadiografias'),
        Path('/content/drive/MyDrive/CodexProjects/dataRadiografias'),
        Path.cwd(),
    ])
    for candidate in candidates:
        if (candidate / 'indice_dataset.csv').exists():
            return candidate
    return candidates[0]


PROJECT_ROOT = resolve_project_root()
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f'No existe PROJECT_ROOT={PROJECT_ROOT}. Ajusta esta ruta a tu carpeta real en Google Drive.'
    )

os.chdir(PROJECT_ROOT)
print('Working directory:', Path.cwd())

## 1. Librerias y rutas de resumen

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

ROOT = Path.cwd()

def resolve_existing(*relative_candidates: str) -> Path:
    search_roots = [ROOT, ROOT / 'reports']
    for base in search_roots:
        for rel in relative_candidates:
            candidate = base / rel
            if candidate.exists():
                return candidate
    raise FileNotFoundError(f'No se encontro ninguno de estos archivos: {relative_candidates}')


def resolve_optional(*relative_candidates: str) -> Path | None:
    search_roots = [ROOT, ROOT / 'reports']
    for base in search_roots:
        for rel in relative_candidates:
            candidate = base / rel
            if candidate.exists():
                return candidate
    return None


BITACORA_PATH = resolve_optional('docs/bitacora_proyecto_radiografias.md')
OUTPUT_DIR = ROOT / 'analysis_outputs' / 'final_project_summary_thoracolumbar'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

POST_V1_PATH = resolve_optional(
    'analysis_outputs/thoracolumbar_postprocess_anatomical_explained/postprocess_experiment_summary.csv'
)
POST_V2_PATH = resolve_optional(
    'analysis_outputs/thoracolumbar_postprocess_anatomical_v2_explained/postprocess_v2_experiment_summary.csv'
)
RANGE_PATH = resolve_optional(
    'analysis_outputs/visible_range_estimator_thoracolumbar_explained/visible_range_experiment_summary.csv'
)
LAST_PATH = resolve_optional(
    'analysis_outputs/last_visible_estimator_thoracolumbar_explained/last_visible_experiment_summary.csv'
)

print('BITACORA_PATH:', BITACORA_PATH)
print('OUTPUT_DIR:', OUTPUT_DIR)

## 2. Decisiones principales del proyecto

Estas decisiones no fueron arbitrarias; surgieron del comportamiento del dataset y
de los errores observados durante la exploracion.

In [ ]:
decision_rows = [
    {
        'decision': 'Excluir vertebras cervicales',
        'por_que': 'El objetivo clinico y el consejo experto enfocaron el problema en thoracic + lumbar.',
        'impacto': 'Se redujo la complejidad del problema y se concentro el aprendizaje en la region util.',
    },
    {
        'decision': 'Usar estrategia en cascada',
        'por_que': 'La columna completa es mas facil de localizar que separar directamente cada vertebra en la radiografia completa.',
        'impacto': 'El modelo multiclase trabajo dentro de una ROI mas limpia y anatomica.',
    },
    {
        'decision': 'Adoptar subset partial',
        'por_que': 'Las imagenes parciales representan un caso real y aumentan el numero de muestras disponibles.',
        'impacto': 'Se mejoro el entrenamiento, pero aparecio el problema de sobreprediccion en secuencias parciales.',
    },
    {
        'decision': 'No usar postproceso anatomico v1',
        'por_que': 'El recorte duro destruyo casi toda la segmentacion util.',
        'impacto': 'Se descarto como solucion final y quedo solo como evidencia de exploracion.',
    },
    {
        'decision': 'Mantener postproceso v2 como referencia',
        'por_que': 'Mejoro monotonia anatomica, pero apenas redujo sobreprediccion.',
        'impacto': 'Sirvio para confirmar que la solucion real debia estimar mejor el rango visible.',
    },
    {
        'decision': 'Crear visible-range estimator',
        'por_que': 'La principal falla era extender la secuencia mas alla de lo visible.',
        'impacto': 'Se logro una primera mejora real sobre raw.',
    },
    {
        'decision': 'Especializar el modelo en last_visible_idx',
        'por_que': 'El analisis demostro que la primera vertebra visible ya estaba casi resuelta y el error real estaba en la ultima.',
        'impacto': 'Se obtuvo la mejor version actual del pipeline.',
    },
]
decisions_df = pd.DataFrame(decision_rows)
display(decisions_df)

## 3. Arquitectura final seleccionada

La arquitectura final no es el primer intento del proyecto, sino el resultado de
varias iteraciones guiadas por evidencia.

In [ ]:
architecture_df = pd.DataFrame([
    {
        'etapa': '1. Binary spine localization',
        'entrada': 'Radiografia completa',
        'salida': 'ROI espinal',
        'razon': 'Reducir fondo y enfocar la siguiente etapa en la columna.',
    },
    {
        'etapa': '2. Multiclase thoracolumbar segmentation',
        'entrada': 'ROI espinal',
        'salida': 'Mascara vertebral T1..T12 + L1..L5',
        'razon': 'Identificar vertebras dentro de una zona ya localizada.',
    },
    {
        'etapa': '3. Last visible estimator',
        'entrada': 'ROI + features anatomicas derivadas de la prediccion multiclase',
        'salida': 'Ultima vertebra visible estimada',
        'razon': 'Corregir el principal fallo del pipeline: sobreextender la secuencia.',
    },
    {
        'etapa': '4. Anatomical clipping',
        'entrada': 'Mascara multiclase + ultima vertebra visible',
        'salida': 'Mascara final corregida',
        'razon': 'Reducir etiquetas extra y mejorar coherencia anatomica final.',
    },
])
display(architecture_df)

## 4. Cronologia de experimentos relevantes

Esta tabla resume el aprendizaje del proyecto, no solo los resultados finales.

In [ ]:
experiment_rows = [
    {
        'stage': 'Binary localization',
        'variant': 'binary_spine_thoracolumbar',
        'key_result': 'test_dice=0.8440',
        'status': 'accepted',
        'comment': 'Suficiente para funcionar como primera etapa de cascada.',
    },
    {
        'stage': 'Cascade multiclass',
        'variant': 'initial',
        'key_result': 'test_macro_dice_fg=0.2289',
        'status': 'superseded',
        'comment': 'Primera version usable, pero claramente insuficiente.',
    },
    {
        'stage': 'Cascade multiclass',
        'variant': 'reinforced',
        'key_result': 'test_macro_dice_fg=0.2730',
        'status': 'superseded',
        'comment': 'Mejora intermedia.',
    },
    {
        'stage': 'Cascade multiclass',
        'variant': 'partial explained',
        'key_result': 'test_macro_dice_fg=0.2854',
        'status': 'reference',
        'comment': 'Mejor baseline multiclase.',
    },
    {
        'stage': 'Postprocess',
        'variant': 'anatomical v1',
        'key_result': 'post_macro_dice_fg=0.0010',
        'status': 'rejected',
        'comment': 'Recorte demasiado agresivo, destruye recall.',
    },
    {
        'stage': 'Postprocess',
        'variant': 'anatomical v2',
        'key_result': 'post_v2_macro_dice_fg=0.2836',
        'status': 'optional',
        'comment': 'Estabiliza monotonia, pero no resuelve la sobreprediccion.',
    },
    {
        'stage': 'Range estimation',
        'variant': 'visible-range estimator',
        'key_result': 'pred_clip_macro_dice_fg=0.2907',
        'status': 'improved',
        'comment': 'Primera mejora real del clipping guiado por rango visible.',
    },
    {
        'stage': 'Range estimation',
        'variant': 'last-visible estimator',
        'key_result': 'last_pred_clip_macro_dice_fg=0.2962',
        'status': 'best_current',
        'comment': 'Mejor variante actual del pipeline.',
    },
]
experiments_df = pd.DataFrame(experiment_rows)
display(experiments_df)

## 5. Carga de resultados exportados de las etapas finales

Aqui se leen los CSV ya generados por los notebooks del proyecto.

In [ ]:
def read_metric_summary(path: Path, stage_name: str) -> pd.DataFrame:
    if path is None or not path.exists():
        return pd.DataFrame({'metric': [], stage_name: []})
    df = pd.read_csv(path)
    if {'metric', 'value'}.issubset(df.columns):
        return df.rename(columns={'value': stage_name})
    return pd.DataFrame({'metric': [], stage_name: []})


post_v1_df = read_metric_summary(POST_V1_PATH, 'post_v1')
post_v2_df = read_metric_summary(POST_V2_PATH, 'post_v2')
range_df = read_metric_summary(RANGE_PATH, 'range_clip')
last_df = read_metric_summary(LAST_PATH, 'last_clip')

merged_df = post_v1_df
for other in [post_v2_df, range_df, last_df]:
    if merged_df.empty:
        merged_df = other
    elif not other.empty:
        merged_df = merged_df.merge(other, on='metric', how='outer')

display(merged_df)

## 6. Comparacion final de la evolucion del pipeline

Esta comparacion se centra en las variantes que realmente compitieron por entrar al
pipeline final.

In [ ]:
final_compare_df = pd.DataFrame([
    {
        'variant': 'raw_multiclass_baseline',
        'macro_dice_fg': 0.2853897834388499,
        'macro_iou_fg': 0.17201970729676735,
        'macro_dice_lumbar': 0.3559081160710854,
        'mean_extra_count': 3.088888888888889,
        'mean_missing_count': 0.044444444444444446,
    },
    {
        'variant': 'postprocess_v2',
        'macro_dice_fg': 0.28361920444678546,
        'macro_iou_fg': 0.17100686770078544,
        'macro_dice_lumbar': 0.35732547145290133,
        'mean_extra_count': 2.9555555555555557,
        'mean_missing_count': 0.37777777777777777,
    },
    {
        'variant': 'visible_range_pred_clip',
        'macro_dice_fg': 0.2906795300717845,
        'macro_iou_fg': 0.17607368184386776,
        'macro_dice_lumbar': 0.373893254623063,
        'mean_extra_count': 2.3777777777777778,
        'mean_missing_count': 0.2,
    },
    {
        'variant': 'last_visible_pred_clip',
        'macro_dice_fg': 0.29615453539162606,
        'macro_iou_fg': 0.18035852781892955,
        'macro_dice_lumbar': 0.38659036638300265,
        'mean_extra_count': 1.6,
        'mean_missing_count': 0.3333333333333333,
    },
    {
        'variant': 'oracle_clip_reference',
        'macro_dice_fg': 0.316585840687472,
        'macro_iou_fg': 0.19648882184509034,
        'macro_dice_lumbar': 0.4471777965806595,
        'mean_extra_count': 0.022222222222222223,
        'mean_missing_count': 0.022222222222222223,
    },
])
display(final_compare_df.sort_values('macro_dice_fg', ascending=False))

## 7. Argumento de seleccion de arquitectura

Esta es la justificacion tecnica de por que se elige `last_visible_pred_clip` como
mejor version actual del pipeline.

In [ ]:
rationale_df = pd.DataFrame([
    {
        'argumento': 'Mejor equilibrio entre mejora y estabilidad',
        'detalle': 'Supera a raw, visible-range clip y postprocess v2 sin colapsar la segmentacion.',
    },
    {
        'argumento': 'Ataca el cuello de botella correcto',
        'detalle': 'El proyecto mostro que el problema principal estaba en la ultima vertebra visible, no en la primera.',
    },
    {
        'argumento': 'Mejora concreta en region lumbar',
        'detalle': 'La mayor sobreprediccion estaba hacia la zona lumbar y alli se observa el mayor beneficio.',
    },
    {
        'argumento': 'Reduccion fuerte de etiquetas extra',
        'detalle': 'Baja mean_extra_count de 3.09 a 1.60, mejor que la etapa de range estimator anterior.',
    },
    {
        'argumento': 'Sigue siendo explicable',
        'detalle': 'La arquitectura final tiene una cadena de decisiones anatomicas faciles de defender en el documento final.',
    },
])
display(rationale_df)

## 8. Recomendaciones finales del proyecto

In [ ]:
recommendations_df = pd.DataFrame([
    {
        'tipo': 'Pipeline final',
        'recomendacion': 'Usar binary -> multiclass -> last_visible_estimator -> clipping como pipeline actual recomendado.',
    },
    {
        'tipo': 'Uso practico',
        'recomendacion': 'Apoyarse en el notebook final de inferencia para procesar nuevas radiografias.',
    },
    {
        'tipo': 'Validacion futura',
        'recomendacion': 'Evaluar el pipeline sobre un conjunto externo o prospectivo si se quiere mayor robustez.',
    },
    {
        'tipo': 'Mejora futura',
        'recomendacion': 'Si se quiere seguir optimizando, el siguiente paso razonable seria una calibracion o ensemble del last_visible estimator.',
    },
])
display(recommendations_df)

## 9. Exportacion del resumen final

Esta celda guarda tablas utiles para el documento final.

In [ ]:
decisions_path = OUTPUT_DIR / 'final_decisions_table.csv'
architecture_path = OUTPUT_DIR / 'final_architecture_table.csv'
experiments_path = OUTPUT_DIR / 'final_experiments_table.csv'
compare_path = OUTPUT_DIR / 'final_pipeline_comparison.csv'
rationale_path = OUTPUT_DIR / 'final_rationale_table.csv'
recommendations_path = OUTPUT_DIR / 'final_recommendations_table.csv'

decisions_df.to_csv(decisions_path, index=False)
architecture_df.to_csv(architecture_path, index=False)
experiments_df.to_csv(experiments_path, index=False)
final_compare_df.to_csv(compare_path, index=False)
rationale_df.to_csv(rationale_path, index=False)
recommendations_df.to_csv(recommendations_path, index=False)

if BITACORA_PATH.exists():
    print('Bitacora disponible en:', BITACORA_PATH)

print('Guardado:', decisions_path)
print('Guardado:', architecture_path)
print('Guardado:', experiments_path)
print('Guardado:', compare_path)
print('Guardado:', rationale_path)
print('Guardado:', recommendations_path)